**dependencies**


In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

**data collection**

In [8]:
# lead the data to pd dataframe
titanic_data = pd.read_csv('/content/train.csv')

In [20]:
# check. missing values in each column
titanic_data.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


**handle missing values**

In [13]:
# drop the cabin column
titanic_data = titanic_data.drop(columns='Cabin', axis=1)
# pandas 3 -> titanic_data['Age'].fillna(titanic_data['Age'].mean(), inplace=True)

In [15]:
# fill age values with mean
titanic_data['Age'].fillna(titanic_data['Age'].mean(), inplace=True)

/tmp/ipykernel_215/1266454042.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic_data['Age'].fillna(titanic_data['Age'].mean(), inplace=True)


In [17]:
# fill embarked column values
print(titanic_data['Embarked'].mode())

0    S
Name: Embarked, dtype: object


In [18]:
print(titanic_data['Embarked'].mode()[0])   # most popular

S


In [19]:
titanic_data['Embarked'].fillna(titanic_data['Embarked'].mode()[0], inplace=True)

/tmp/ipykernel_215/3993763136.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic_data['Embarked'].fillna(titanic_data['Embarked'].mode()[0], inplace=True)


**Feature engineering**

In [155]:
# extract titles
titanic_data['Title'] = titanic_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Group rare titles together
titanic_data['Title'] = titanic_data['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
titanic_data['Title'] = titanic_data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Encode to numbers
titanic_data['Title'] = titanic_data['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

In [156]:
# family size
titanic_data['FamilySize'] = titanic_data['SibSp'] + titanic_data['Parch'] + 1

In [157]:
# is alone
titanic_data['IsAlone'] = (titanic_data['FamilySize'] == 1).astype(int)

**Data Analysis**

**visualize data**

In [23]:
sns.set()

**training: encoding the categorical columns**

In [158]:
# make values numerical : f-> 1 m-> 0, etc
titanic_data['Sex'].value_counts()

,count
Sex,
0,577
1,314


In [159]:
# convert
titanic_data['Sex'].replace({'female':1,'male':0}, inplace=True)
titanic_data['Embarked'].replace({'S':0,'C':1, 'Q':2}, inplace=True)

/tmp/ipykernel_215/2833413902.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  titanic_data['Sex'].replace({'female':1,'male':0}, inplace=True)
/tmp/ipykernel_215/2833413902.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

**separate target column form feature column**

In [160]:
X = titanic_data.drop(columns=['PassengerId', 'Name', 'Ticket', 'Survived', 'SibSp', 'Parch'], axis=1)
Y = titanic_data['Survived']

In [63]:
print(X)

     Pclass  Sex        Age     Fare  Embarked  Title  FamilySize  IsAlone
0         3    0  22.000000   7.2500         0      0           2        0
1         1    1  38.000000  71.2833         1      2           2        0
2         3    1  26.000000   7.9250         0      1           1        1
3         1    1  35.000000  53.1000         0      2           2        0
4         3    0  35.000000   8.0500         0      0           1        1
..      ...  ...        ...      ...       ...    ...         ...      ...
886       2    0  27.000000  13.0000         0      4           1        1
887       1    1  19.000000  30.0000         0      1           1        1
888       3    1  29.699118  23.4500         0      1           4        0
889       1    0  26.000000  30.0000         1      0           1        1
890       3    0  32.000000   7.7500         2      0           1        1

[891 rows x 8 columns]


In [44]:
print(Y)

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64


**split data into test and train**

In [161]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.17, random_state=2)

In [162]:
print(X.shape,X_train.shape, X_test.shape)

(891, 8) (739, 8) (152, 8)


**model training** | logistic regression

In [143]:
# model = model = LogisticRegression()
# model = RandomForestClassifier()   best performin
# model = GradientBoostingClassifier()
# model = SVC().   worst pereforming

**evaluate the model** | Accuracy Score

In [163]:
model = RandomForestClassifier(
    n_estimators=100,   # number of trees
    max_depth=5,        # limits how deep each tree grows
    min_samples_split=10,  # min samples needed to split a node
    min_samples_leaf=4,    # min samples required at each leaf
    random_state=2
)

model.fit(X_train, Y_train)

print(f"Training Accuracy: {model.score(X_train, Y_train):.4f}")
print(f"Testing Accuracy: {model.score(X_test, Y_test):.4f}")

Training Accuracy: 0.8498
Testing Accuracy: 0.8158


---


# **TEST**

In [164]:
kaggle_test = pd.read_csv('/content/test.csv')

In [165]:
# Same cleaning as you did on training data
kaggle_test['Age'].fillna(kaggle_test['Age'].median(), inplace=True)
kaggle_test['Fare'].fillna(kaggle_test['Fare'].median(), inplace=True)
kaggle_test['Embarked'].fillna(kaggle_test['Embarked'].mode()[0], inplace=True)

# Same feature engineering
kaggle_test['Title'] = kaggle_test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
kaggle_test['Title'] = kaggle_test['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
kaggle_test['Title'] = kaggle_test['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
kaggle_test['Title'] = kaggle_test['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

kaggle_test['FamilySize'] = kaggle_test['SibSp'] + kaggle_test['Parch'] + 1
kaggle_test['IsAlone'] = (kaggle_test['FamilySize'] == 1).astype(int)

# Same Sex encoding
kaggle_test['Sex'] = kaggle_test['Sex'].replace({'female': 1, 'male': 0})

# Same Embarked encoding
kaggle_test['Embarked'] = kaggle_test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

/tmp/ipykernel_215/445517607.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  kaggle_test['Age'].fillna(kaggle_test['Age'].median(), inplace=True)
/tmp/ipykernel_215/445517607.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpl

In [166]:
X_kaggle = kaggle_test.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch'])

In [167]:
print("X_train columns:", X_train.columns.tolist())
print("X_kaggle columns:", X_kaggle.columns.tolist())

X_train columns: ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']
X_kaggle columns: ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']


In [168]:
predictions = model.predict(X_kaggle)

In [169]:
submission = pd.DataFrame({
    'PassengerId': kaggle_test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)
print(submission.head())

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
